[Home](../../README.md)

### __Data Wrangling__

This Jupyter Notepad demonstrates the different processes applied to the data to prepare it for feature engineering and model training.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### __Loading the required dependencies__

In [96]:
# Import frameworks
import pandas as pd

####  __Storing the data as a local variable__

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [97]:
data_frame = pd.read_csv("2.1.2.domain_properties.csv")

#### __Dealing with null values__

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [98]:
data_frame.isnull().sum()

price                       0
date_sold                   0
suburb                      0
num_bath                    0
num_bed                     0
num_parking                 0
property_size               0
type                        0
suburb_population           0
suburb_median_income        0
suburb_sqkm                 0
suburb_lat                  0
suburb_lng                  0
suburb_elevation            0
cash_rate                   0
property_inflation_index    0
km_from_cbd                 0
dtype: int64

If there was any null data it could be removed using:
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Since there is no null data in any of the columns, we do not have to worry about removing anything.

#### __Removing Duplicates__

Duplicate data can have detrimental effects on the machine learning models and outcomes, such as reducing data diversity and representativeness, which can lead to overfitting or biased models.

The `duplicated().sum()` method call returns the count of duplicate rows in the data frame.

In [99]:
data_frame.duplicated().sum()

np.int64(0)

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [100]:
data_frame = data_frame.drop_duplicates()
data_frame.duplicated().sum()

np.int64(0)

However since the `duplicated().sum()` method call returned a count of 0 duplicates, we do not have to worry about removing any.

#### __Replacing data__

We can run a lambda function on a column to modify its values. I will convert the type to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [101]:
data_frame['type'] = data_frame['type'].apply(lambda x: x.lower())
data_frame['type'].head()

0          house
1          house
2          house
3          house
4    vacant land
Name: type, dtype: str

We can check that there are no data entry errors by the `unique()` method call.

In [102]:
data_frame['type'].unique()

<StringArray>
[                        'house',                   'vacant land',
                     'townhouse',       'apartment / unit / flat',
                 'semi-detached',              'new house & land',
                        'duplex',                         'villa',
                      'new land',                       'terrace',
                        'studio',                'block of units',
              'development site',          'acreage / semi-rural',
 'new apartments / off the plan',                         'rural']
Length: 16, dtype: str

Since all the types are in order, we do not need to replace any of them

#### __Deleting features and rows__

Since this won't be a type of classification model, the first step when deleting features is to delete features that use data type we can't use to predict anything with (e.g. strings, DD/MM/YYYY format).

In this dataset, these features are __date sold__, __suburb__, and __type__. However, we won't delete __type__ yet because it will be used for something later in the wrangling process.

In [103]:
data_frame = data_frame.drop(columns=["date_sold", "suburb"])

After looking at the data in the data preview notebook, I came to the decision to get rid of columns that had no correlation and would obviously perform poorly if used them to train the model on. I came to the conclusion that these columns were __cash rate__ and __property inflation index__.  

In [104]:
data_frame = data_frame.drop(columns=["cash_rate", "property_inflation_index"])

> [__Justification__]
> Though the date sold, property inflation index and cash rate are typically important in estimating the price of a property, I decided to remove them from the dataset since the features reflect temporary economic conditions. This model aims to predict house prices based on features of the property and including the date sold, property inflation index and cash rate features could cause the model to learn short term fluctuations in the property market.

Next I will remove features that have very little to no correlation and contain no predictive value for estimating house prices. These appear to be the __suburb latitude__ and __suburb longitude__.

In [105]:
data_frame = data_frame.drop(columns=["suburb_lat", "suburb_lng"])

Now I will move onto removing unwanted rows in the data. The __type__ column shows listings for different types of properties that have been sold. Since we are trying to predict the price of a house in NSW, we want to remove property types that do not include any actual buildings or something a typical first home owner wouldn't purchase such as vacant land, development sites, block of units, etc.

In [106]:
data_frame = data_frame[data_frame["type"] != "vacant land"]

In [107]:
data_frame = data_frame[data_frame["type"] != "new land"]

In [108]:
data_frame = data_frame[data_frame["type"] != "development site"]

In [109]:
data_frame = data_frame[data_frame["type"] != "block of units"]

Now __type__ can be removed.

In [110]:
data_frame = data_frame.drop(columns=["type"])

#### __Removing outliers__

Outliers can skew analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [111]:
# Price outliers
print(data_frame['price'].describe())
Q1 = data_frame['price'].quantile(0.25)
Q3 = data_frame['price'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a price above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')

count    1.095000e+04
mean     1.678275e+06
std      1.286159e+06
min      2.725000e+05
25%      1.010000e+06
50%      1.392250e+06
75%      2.020000e+06
max      6.000000e+07
Name: price, dtype: float64
Outliers are a price above 3535000.0 or below -505000.0


In [112]:
data_frame = data_frame[(data_frame['price'] >= Q1 - 1.5 * IQR) & (data_frame['price'] <= Q3 + 1.5 * IQR)]
print(data_frame['price'].describe())

count    1.038300e+04
mean     1.482837e+06
std      6.928647e+05
min      2.725000e+05
25%      1.000000e+06
50%      1.340000e+06
75%      1.901000e+06
max      3.530000e+06
Name: price, dtype: float64


To make the process of removing outliers easier and more efficient, the code below uses a loop to iterate through a list and remove the outliers from the features. 

In [113]:
remove_outliers = [
    "num_bath",
    "num_bed",
    "num_parking",
    "property_size",
    "suburb_population",
    "suburb_median_income",
    "suburb_sqkm",
    "suburb_elevation",
    "km_from_cbd",
]

for feature in remove_outliers:
    Q1 = data_frame[feature].quantile(0.25)
    Q3 = data_frame[feature].quantile(0.75)
    IQR = Q3 - Q1
    data_frame = data_frame[(data_frame[feature] >= Q1 - 1.5 * IQR) & (data_frame[feature] <= Q3 + 1.5 * IQR)]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.00000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,1.860242,3.531434,1.694067,550.545750,8359.55062,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.693102,0.863299,0.717807,238.155784,5491.09087,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,2.000000,0.000000,7.000000,22.00000,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,1.000000,3.000000,1.000000,392.000000,3998.75000,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,2.000000,4.000000,2.000000,567.500000,7353.00000,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,2.000000,4.000000,2.000000,696.000000,11674.00000,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,3.000000,5.000000,3.000000,1214.000000,24541.00000,64532.000000,12.516000,137.000000,84.790000


#### __Scaling features to a common range__

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [114]:
scale_feature = 'num_bath'

#the minimum value with space for outliers
MIN_NUM_BATH = 0

#the maximum value with space for outliers
MAX_NUM_BATH = 6

#scale features
data_frame[scale_feature] = [(X - MIN_NUM_BATH) / (MAX_NUM_BATH - MIN_NUM_BATH) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.00000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,3.531434,1.694067,550.545750,8359.55062,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.863299,0.717807,238.155784,5491.09087,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,2.000000,0.000000,7.000000,22.00000,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,3.000000,1.000000,392.000000,3998.75000,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,4.000000,2.000000,567.500000,7353.00000,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,4.000000,2.000000,696.000000,11674.00000,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,5.000000,3.000000,1214.000000,24541.00000,64532.000000,12.516000,137.000000,84.790000


In [115]:
scale_feature = "num_bed"

# the minimum value with space for outliers
MIN_NUM_BED = 0

# the maximum value with space for outliers
MAX_NUM_BED = 9

# scale features
data_frame[scale_feature] = [(X - MIN_NUM_BED) / (MAX_NUM_BED - MIN_NUM_BED) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.00000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,1.694067,550.545750,8359.55062,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.717807,238.155784,5491.09087,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,7.000000,22.00000,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,1.000000,392.000000,3998.75000,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,2.000000,567.500000,7353.00000,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,2.000000,696.000000,11674.00000,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,3.000000,1214.000000,24541.00000,64532.000000,12.516000,137.000000,84.790000


In [116]:
scale_feature = "num_parking"

# the minimum value with space for outliers
MIN_NUM_PARKING = 0

# the maximum value with space for outliers
MAX_NUM_PARKING = 6

# scale features
data_frame[scale_feature] = [(X - MIN_NUM_PARKING) / (MAX_NUM_PARKING - MIN_NUM_PARKING) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.00000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,550.545750,8359.55062,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,238.155784,5491.09087,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,7.000000,22.00000,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,392.000000,3998.75000,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,567.500000,7353.00000,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,696.000000,11674.00000,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1214.000000,24541.00000,64532.000000,12.516000,137.000000,84.790000


In [117]:
scale_feature = "property_size"

# the minimum value with space for outliers
MIN_PROPERTY_SIZE = 2

# the maximum value with space for outliers
MAX_PROPERTY_SIZE = 1200

# scale features
data_frame[scale_feature] = [(X - MIN_PROPERTY_SIZE) / (MAX_PROPERTY_SIZE - MIN_PROPERTY_SIZE) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.00000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,8359.55062,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,5491.09087,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,22.00000,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,3998.75000,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,7353.00000,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,11674.00000,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,24541.00000,64532.000000,12.516000,137.000000,84.790000


In [118]:
scale_feature = "suburb_population"

# the minimum value with space for outliers
MIN_SUBURB_POPULATION = 18

# the maximum value with space for outliers
MAX_SUBURB_POPULATION = 25000

# scale features
data_frame[scale_feature] = [(X - MIN_SUBURB_POPULATION) / (MAX_SUBURB_POPULATION - MIN_SUBURB_POPULATION) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,0.333902,38377.550177,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,0.219802,9273.653721,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,0.000160,14248.000000,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,0.159345,31824.000000,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,0.293611,37336.000000,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,0.466576,44668.000000,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,0.981627,64532.000000,12.516000,137.000000,84.790000


In [119]:
scale_feature = "suburb_median_income"

# the minimum value with space for outliers
MIN_SUBURB_MEDIAN_INCOME = 14000

# the maximum value with space for outliers
MAX_SUBURB_MEDIAN_INCOME = 67000

# scale features
data_frame[scale_feature] = [(X - MIN_SUBURB_MEDIAN_INCOME) / (MAX_SUBURB_MEDIAN_INCOME - MIN_SUBURB_MEDIAN_INCOME) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,0.333902,0.459954,3.778584,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,0.219802,0.174975,2.736112,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,0.000160,0.004679,0.089000,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,0.159345,0.336302,1.730000,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,0.293611,0.440302,3.049000,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,0.466576,0.578642,5.117000,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,0.981627,0.953434,12.516000,137.000000,84.790000


In [120]:
scale_feature = "suburb_sqkm"

# the minimum value with space for outliers
MIN_SUBURB_SQKM = 0.050000

# the maximum value with space for outliers
MAX_SUBURB_SQKM = 11

# scale features
data_frame[scale_feature] = [(X - MIN_SUBURB_SQKM) / (MAX_SUBURB_SQKM - MIN_SUBURB_SQKM) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,0.333902,0.459954,0.340510,43.154073,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,0.219802,0.174975,0.249873,31.969545,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,0.000160,0.004679,0.003562,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,0.159345,0.336302,0.153425,18.000000,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,0.293611,0.440302,0.273881,35.000000,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,0.466576,0.578642,0.462740,59.000000,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,0.981627,0.953434,1.138447,137.000000,84.790000


In [121]:
scale_feature = "suburb_elevation"

# the minimum value with space for outliers
MIN_SUBURB_ELEVATION = 0

# the maximum value with space for outliers
MAX_SUBURB_ELEVATION = 121

# scale features
data_frame[scale_feature] = [(X - MIN_SUBURB_ELEVATION) / (MAX_SUBURB_ELEVATION - MIN_SUBURB_ELEVATION) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,0.333902,0.459954,0.340510,0.356645,28.017900
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,0.219802,0.174975,0.249873,0.264211,18.790235
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,0.000160,0.004679,0.003562,0.000000,0.310000
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,0.159345,0.336302,0.153425,0.148760,13.340000
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,0.293611,0.440302,0.273881,0.289256,22.430000
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,0.466576,0.578642,0.462740,0.487603,42.740000
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,0.981627,0.953434,1.138447,1.132231,84.790000


In [122]:
scale_feature = "km_from_cbd"

# the minimum value with space for outliers
MIN_KM_FROM_CBD = 0.15

# the maximum value with space for outliers
MAX_KM_FROM_CBD = 100

# scale features
data_frame[scale_feature] = [(X - MIN_KM_FROM_CBD) / (MAX_KM_FROM_CBD - MIN_KM_FROM_CBD) for X in data_frame[scale_feature]]

data_frame.describe()

,price,num_bath,num_bed,num_parking,property_size,suburb_population,suburb_median_income,suburb_sqkm,suburb_elevation,km_from_cbd
count,6.776000e+03,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000,6776.000000
mean,1.399862e+06,0.310040,0.392382,0.282345,0.457885,0.333902,0.459954,0.340510,0.356645,0.279098
std,6.575688e+05,0.115517,0.095922,0.119635,0.198794,0.219802,0.174975,0.249873,0.264211,0.188185
min,2.750000e+05,0.000000,0.222222,0.000000,0.004174,0.000160,0.004679,0.003562,0.000000,0.001602
25%,9.000000e+05,0.166667,0.333333,0.166667,0.325543,0.159345,0.336302,0.153425,0.148760,0.132098
50%,1.280000e+06,0.333333,0.444444,0.333333,0.472037,0.293611,0.440302,0.273881,0.289256,0.223135
75%,1.800000e+06,0.333333,0.444444,0.333333,0.579299,0.466576,0.578642,0.462740,0.487603,0.426540
max,3.530000e+06,0.500000,0.555556,0.500000,1.011686,0.981627,0.953434,1.138447,1.132231,0.847672


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### __Saving the wrangled data to CSV__

In [123]:
data_frame.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_data.csv', index=False)